# dsfb-turbine Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-turbine/notebooks/dsfb_turbine_colab.ipynb)

This notebook is designed for a fresh Colab runtime. It clones the repository, installs Rust, downloads the public NASA C-MAPSS benchmark archive referenced by the crate, runs `cargo run --example cmapss_eval -- --require-real-data` against the benchmark files, stages generated figures, builds a combined PDF of the figures, and packages the run artifacts into a zip for download.

Purpose:

- This is the std-backed reproducibility and artifact-packaging surface for the crate, not the embedded `no_std` core API.
- It demonstrates how DSFB is run on the public benchmark archive when the residual-interpretation layer is calibrated for that benchmark setting.
- It is not a head-to-head benchmark competition notebook and should not be read as a claim that DSFB replaces incumbent probabilistic EHM, GPA, or PHM methods.
- The role of DSFB here is to augment existing residual-producing systems by attaching deterministic, typed, human-readable structural interpretation to those residual streams.

Empirical scope:

- Full FD001/FD002/FD003 paper reproduction requires `train_FD001.txt`, `train_FD002.txt`, and `train_FD003.txt` to be present after archive extraction.
- The notebook is configured to fail closed if those benchmark files are not recovered. It does not intentionally proceed to the crate's synthetic fallback path.
- NASA C-MAPSS is a benchmark simulation dataset, not field-engine telemetry. The notebook therefore runs the real benchmark archive used by the crate, not a real-world flight dataset in the strict field-data sense.
- Notebook outputs are written under `crates/dsfb-turbine/colab_output/` inside the fresh clone for each run.


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

REPO_URL = "https://github.com/infinityabundance/dsfb.git"
NOTEBOOK_URL = "https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-turbine/notebooks/dsfb_turbine_colab.ipynb"
NASA_CMAPSS_URL = "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"

REPO_DIR = Path("/content/dsfb")
CRATE_DIR = REPO_DIR / "crates" / "dsfb-turbine"
RUN_DIR = CRATE_DIR / "colab_output"
DATA_DIR = RUN_DIR / "data"
ARTIFACTS_DIR = RUN_DIR / "artifacts"
FIGURES_DIR = RUN_DIR / "figures"
PAGE_PDF_DIR = RUN_DIR / "figure_pages"
DOWNLOADS_DIR = RUN_DIR / "downloads"
COMBINED_PDF_PATH = DOWNLOADS_DIR / "dsfb_turbine_all_figures.pdf"
ZIP_PATH = DOWNLOADS_DIR / "dsfb_turbine_artifacts.zip"
DATASET_NAMES = ("train_FD001.txt", "train_FD002.txt", "train_FD003.txt")

def run(cmd, cwd=None, check=True, env=None):
    print("$", " ".join(cmd))
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        env=merged_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}: {' '.join(cmd)}")
    return completed

def copy_first_match(search_root, filename, target_dir):
    matches = sorted(search_root.rglob(filename))
    if not matches:
        return None
    target_path = target_dir / filename
    shutil.copy2(matches[0], target_path)
    return target_path

def extract_zip_tree(zip_path, extract_root):
    pending = [Path(zip_path)]
    seen = set()
    while pending:
        current = pending.pop()
        current = Path(current)
        if current in seen:
            continue
        seen.add(current)
        target_dir = extract_root / current.stem
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(current) as zf:
            zf.extractall(target_dir)
        for nested in sorted(target_dir.rglob("*")):
            if nested.is_file() and nested.suffix.lower() == ".zip":
                pending.append(nested)

print("Notebook:", NOTEBOOK_URL)
print("Repo:", REPO_URL)


In [ ]:
run(["apt-get", "update", "-qq"])
run([
    "apt-get", "install", "-y",
    "git", "curl", "zip", "unzip",
    "libcairo2", "libpango-1.0-0", "libpangocairo-1.0-0", "libgdk-pixbuf-2.0-0",
])
run([sys.executable, "-m", "pip", "install", "-q", "cairosvg==2.7.1", "pypdf==4.3.1"])

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

cargo_bin = Path.home() / ".cargo" / "bin" / "cargo"
if not cargo_bin.exists():
    run([
        "bash", "-lc",
        "curl https://sh.rustup.rs -sSf | sh -s -- -y --profile minimal --default-toolchain stable",
    ])

os.environ["PATH"] = f"{Path.home() / '.cargo' / 'bin'}:{os.environ['PATH']}"
run(["cargo", "--version"])

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

for path in (DATA_DIR, ARTIFACTS_DIR, FIGURES_DIR, PAGE_PDF_DIR, DOWNLOADS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Crate directory:", CRATE_DIR)
print("Run directory:", RUN_DIR)


In [ ]:
archive_path = DATA_DIR / "cmapss_public.zip"
extract_dir = DATA_DIR / "cmapss_extracted"

print("Downloading NASA C-MAPSS benchmark archive from:")
print(NASA_CMAPSS_URL)
print("Dataset note: C-MAPSS is a public NASA benchmark simulation dataset, not field telemetry.")
urlretrieve(NASA_CMAPSS_URL, archive_path)
extract_zip_tree(archive_path, extract_dir)
for dataset_name in DATASET_NAMES:
    copied_path = copy_first_match(extract_dir, dataset_name, DATA_DIR)
    print(f"{dataset_name}: {copied_path}")

available = sorted(path.name for path in DATA_DIR.glob("train_*.txt"))
missing = [name for name in DATASET_NAMES if name not in available]

print("Available dataset files:", available)
if missing:
    raise RuntimeError(
        "Required NASA C-MAPSS files were not recovered after recursive extraction: "
        + ", ".join(missing)
        + ". Notebook aborted to avoid the synthetic fallback path."
    )
else:
    print("All required files for the full FD001/FD002/FD003 evaluation are present.")


In [ ]:
run(
    ["cargo", "run", "--example", "cmapss_eval", "--", "--require-real-data", str(DATA_DIR), str(ARTIFACTS_DIR)],
    cwd=CRATE_DIR,
)

if (ARTIFACTS_DIR / "report_synthetic.txt").exists():
    raise RuntimeError("Synthetic fallback output was produced unexpectedly; benchmark run requirement was not met.")

expected_reports = [
    ARTIFACTS_DIR / "report_fd001_recommended.txt",
    ARTIFACTS_DIR / "report_fd002_regime.txt",
    ARTIFACTS_DIR / "report_fd003.txt",
]
missing_reports = [path.name for path in expected_reports if not path.exists()]
if missing_reports:
    raise RuntimeError("Benchmark run completed without all expected reports: " + ", ".join(missing_reports))

artifact_files = sorted(path.name for path in ARTIFACTS_DIR.iterdir() if path.is_file())
print("Generated artifact files:")
for name in artifact_files:
    print(" -", name)


In [ ]:
import cairosvg
from pypdf import PdfReader, PdfWriter

svg_paths = sorted(ARTIFACTS_DIR.glob("*.svg"))
if not svg_paths:
    raise RuntimeError("No SVG figures were generated. The packaging step expects at least one SVG artifact.")

for svg_path in svg_paths:
    shutil.copy2(svg_path, FIGURES_DIR / svg_path.name)

page_pdf_paths = []
for svg_path in svg_paths:
    page_pdf_path = PAGE_PDF_DIR / f"{svg_path.stem}.pdf"
    cairosvg.svg2pdf(url=str(svg_path), write_to=str(page_pdf_path))
    page_pdf_paths.append(page_pdf_path)

writer = PdfWriter()
for page_pdf_path in page_pdf_paths:
    reader = PdfReader(str(page_pdf_path))
    for page in reader.pages:
        writer.add_page(page)

with open(COMBINED_PDF_PATH, "wb") as handle:
    writer.write(handle)

print("Staged SVG figures:")
for svg_path in svg_paths:
    print(" -", FIGURES_DIR / svg_path.name)
print("Combined figure PDF:", COMBINED_PDF_PATH)


In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for root in (ARTIFACTS_DIR, FIGURES_DIR):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(RUN_DIR))
    archive.write(COMBINED_PDF_PATH, COMBINED_PDF_PATH.relative_to(RUN_DIR))

print("Artifact zip:", ZIP_PATH)
print("Run root:", RUN_DIR)

try:
    from google.colab import files
    print("Starting Colab downloads for the combined PDF and the artifact zip.")
    files.download(str(COMBINED_PDF_PATH))
    files.download(str(ZIP_PATH))
except Exception as exc:
    print(f"Automatic Colab downloads were skipped: {exc}")
    print(f"Download manually from:\n  {COMBINED_PDF_PATH}\n  {ZIP_PATH}")
